# 02b · Holder snapshots for placebo and negative-control dates

Inputs: `holder_snapshot`, `key_exposure`, `address_key` (02); `placebo_days` (03)

Output: `holder_snapshot_placebo`

Holder snapshot at the close of t−1 for every date in `placebo_days`: randomization pools `ri_pool` and `ri_pool_sym30`, and negative controls `nonquantum_news`. Columns match `holder_snapshot`, so 05 can `UNION ALL` the two tables.

Unlike 02, only the risk set is kept: balance at t−1 ≥ 0.01 BTC and at least one incoming transfer in the previous 12 months (the 0.1 BTC definition is a subset, selected in 05).

- Snapshot labels: `pl_YYYY-MM-DD` (randomization pools), `nq_YYYY-MM-DD` (non-quantum news days); a date in both pools is built once.
- Dates are built in batches of `SNAP_CHUNK` and merged at the end.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

RISK_MIN_BTC = 0.01           # risk set: minimum balance at t-1 (main definition)
RISK_LOOKBACK_DAYS = 365      # risk set: incoming transfer within the last 12 months
SNAP_CHUNK   = 20             # snapshot dates per batch; lower if "Resources exceeded"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, time, datetime as dt
import pandas as pd

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["utxo_life", "address_key", "key_exposure", "holder_snapshot", "placebo_days"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from 02 and 03 found. Spent so far at list price: $%.2f / $%.0f" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}; set CONFIRM=True and re-run")
        return
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))

def q(sql):
    return client.query(sql).to_dataframe()

## Step 1 · Snapshot dates

Distinct dates from `placebo_days`, each with the pools it belongs to.

In [ ]:
pdays = q(f"SELECT kind, date, snap_date FROM `{DS}.placebo_days`")
pdays["date"] = pd.to_datetime(pdays.date)
pdays["snap_date"] = pd.to_datetime(pdays.snap_date)
snaps = (pdays.groupby(["date", "snap_date"]).kind.apply(lambda s: ",".join(sorted(set(s)))).reset_index()
         .sort_values("date").reset_index(drop=True))
snaps["event"] = [("nq_" if k == "nonquantum_news" else "pl_") + d.strftime("%Y-%m-%d")
                  for d, k in zip(snaps.date, snaps.kind)]
assert snaps.event.is_unique
print(f"{len(snaps)} snapshot dates; by kind:", pdays.kind.value_counts().to_dict())
display(snaps.kind.value_counts())

# must not coincide with a main-event snapshot date
main_snap = q(f"SELECT DISTINCT snap_date FROM `{DS}.holder_snapshot`")
clash = set(pd.to_datetime(main_snap.snap_date)) & set(snaps.snap_date)
assert not clash, f"Coincides with a main-event snapshot date: {clash}"

chunks = [snaps.iloc[i:i + SNAP_CHUNK] for i in range(0, len(snaps), SNAP_CHUNK)]
print(f"Built in {len(chunks)} batches (SNAP_CHUNK = {SNAP_CHUNK})")

## Step 2 · Snapshots

SQL of 02 with the risk-set filter. Groups: Taproot → T; exposed on or before the snapshot date → L; otherwise → C. Six standard types only.

In [ ]:
def snapshot_sql(table, sub):
    snaps_sql = ",\n    ".join(
        f"STRUCT('{r.event}' AS event, DATE '{r.date:%Y-%m-%d}' AS event_date, DATE '{r.snap_date:%Y-%m-%d}' AS snap_date)"
        for r in sub.itertuples())
    return f'''
CREATE OR REPLACE TABLE `{DS}.{table}`
CLUSTER BY event, grp
AS
WITH snaps AS (
  SELECT * FROM UNNEST([
    {snaps_sql}
  ])
),
held AS (
  SELECT s.event, s.event_date, s.snap_date, u.address, u.type, u.sats, u.created_date,
         (u.spent_date IS NULL OR u.spent_date > s.snap_date) AS alive
  FROM `{DS}.utxo_life` u
  CROSS JOIN snaps s
  WHERE u.type IN ('pubkey','pubkeyhash','witness_v0_keyhash','scripthash',
                   'witness_v0_scripthash','witness_v1_taproot')
    AND u.address IS NOT NULL
    AND u.created_date <= s.snap_date
    AND ( u.spent_date IS NULL OR u.spent_date > s.snap_date
          OR u.created_date > DATE_SUB(s.snap_date, INTERVAL 365 DAY) )
),
agg AS (
  SELECT
    a.event, a.event_date, a.snap_date, k.key_id,
    ANY_VALUE(k.key_kind)                          AS key_kind,
    SUM(IF(a.alive, a.sats, 0))                    AS sats,
    COUNTIF(a.alive)                               AS n_utxo,
    COUNT(DISTINCT IF(a.alive, a.address, NULL))   AS n_addr,
    MIN(IF(a.alive, a.created_date, NULL))         AS oldest_utxo_date,
    MAX(IF(a.alive, a.created_date, NULL))         AS newest_utxo_date,
    MAX(a.created_date)                            AS last_in_date,
    ARRAY_AGG(DISTINCT IF(a.alive, a.type, NULL) IGNORE NULLS)         AS types
  FROM held a
  JOIN `{DS}.address_key` k USING (address)
  WHERE k.key_id IS NOT NULL
  GROUP BY a.event, a.event_date, a.snap_date, k.key_id
  HAVING SUM(IF(a.alive, a.sats, 0)) >= {int(round(RISK_MIN_BTC * 1e8))}
     AND MAX(a.created_date) > DATE_SUB(a.snap_date, INTERVAL {RISK_LOOKBACK_DAYS} DAY)
)
SELECT
  g.*,
  e.first_exposed_date,
  e.first_exposed_src,
  CASE WHEN g.key_kind = 't'                       THEN 'T'
       WHEN e.first_exposed_date <= g.snap_date    THEN 'L'
       ELSE 'C' END AS grp
FROM agg g
LEFT JOIN `{DS}.key_exposure` e USING (key_id)
'''

chunk_tables = []
for i, sub in enumerate(chunks, 1):
    name = f"holder_snapshot_placebo_c{i:02d}" if len(chunks) > 1 else "holder_snapshot_placebo"
    chunk_tables.append(name)
    print(f"--- batch {i}/{len(chunks)}: {len(sub)} dates ({sub.date.min():%Y-%m-%d} ... {sub.date.max():%Y-%m-%d})")
    build(name, snapshot_sql(name, sub))

In [ ]:
# merge batch tables (no-op for a single batch)
if len(chunk_tables) > 1:
    missing = [t for t in chunk_tables if not table_rows(t)]
    assert not missing, f"These batches are not built yet: {missing}"
    union = "\nUNION ALL\n".join(f"SELECT * FROM `{DS}.{t}`" for t in chunk_tables)
    build("holder_snapshot_placebo", f'''
CREATE OR REPLACE TABLE `{DS}.holder_snapshot_placebo`
CLUSTER BY event, grp
AS
{union}
''')
    if table_rows("holder_snapshot_placebo"):
        for t in chunk_tables:
            client.delete_table(f"{DS}.{t}", not_found_ok=True)
        print("Batch tables dropped")
print(table_info("holder_snapshot_placebo"))

## Checks

1. Every snapshot date is present
2. Risk-set size per date is comparable to the main-event snapshots under the same filter
3. Group shares (L / C / T) move smoothly over time

In [ ]:
got = q(f'''
SELECT event, snap_date, grp, COUNT(*) AS n_keys, ROUND(SUM(sats)/1e8) AS btc,
       COUNTIF(sats >= 1e7) AS n_keys_ge_0_1
FROM `{DS}.holder_snapshot_placebo`
GROUP BY event, snap_date, grp
''')
got["snap_date"] = pd.to_datetime(got.snap_date)
missing = set(snaps.event) - set(got.event)
print("1) Missing snapshots:", sorted(missing) if missing else "none")
assert not missing

wide = got.pivot_table(index=["event", "snap_date"], columns="grp", values="n_keys", aggfunc="sum").fillna(0).astype(int)
wide_btc = got.pivot_table(index=["event", "snap_date"], columns="grp", values="btc", aggfunc="sum").fillna(0)
print("2) Placebo snapshots: risk-set keys per date (>=0.01 BTC, incoming transfer within 12 months)")
display(wide.sort_index(level="snap_date"))

ref = q(f'''
SELECT event, snap_date, grp, COUNT(*) AS n_keys, ROUND(SUM(sats)/1e8) AS btc
FROM `{DS}.holder_snapshot`
WHERE event != 'latest'
  AND sats >= {int(round(RISK_MIN_BTC * 1e8))}
  AND last_in_date > DATE_SUB(snap_date, INTERVAL {RISK_LOOKBACK_DAYS} DAY)
GROUP BY event, snap_date, grp
''')
ref_w = ref.pivot_table(index=["event", "snap_date"], columns="grp", values="n_keys", aggfunc="sum").fillna(0).astype(int)
print("   Reference: main-event snapshots filtered with the same conditions")
display(ref_w.sort_index(level="snap_date"))

print("3) Share of group L in the risk set, by date")
share = (wide["L"] / wide.sum(axis=1)).rename("L_share").reset_index().sort_values("snap_date")
display(share.round(3))

In [ ]:
for rec in JOBS[-5:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
for t in chunk_tables + ["holder_snapshot_placebo"]:
    n = table_rows(t)
    print(f"{t:32s}", f"{n:,} rows" if n else "not built yet")

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "risk_min_btc": RISK_MIN_BTC, "risk_lookback_days": RISK_LOOKBACK_DAYS,
          "n_snapshots": len(snaps), "kinds": pdays.kind.value_counts().to_dict(),
          "ledger": LEDGER, "table": table_info("holder_snapshot_placebo")}
try:
    report["riskset_by_snapshot"] = wide.reset_index().astype(str).to_dict(orient="records")
    report["riskset_main_events"] = ref_w.reset_index().astype(str).to_dict(orient="records")
except NameError:
    pass
with open("build_report_02b.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
print(json.dumps(report["table"], ensure_ascii=False))
try:
    from google.colab import files
    files.download("build_report_02b.json")
except Exception:
    print("Report saved to build_report_02b.json")